In [ ]:
# Import required Python libraries
import pandas as pd
import numpy as np
!pip install faker
from faker import Faker
import random
from datetime import datetime,timedelta

In [ ]:
# Initialize the Faker library
fake = Faker()

In [ ]:
# Generate Customer Data
num_customers = 10000

# Generate unique random 7-digit CustomerID
customer_ids = np.random.choice(range(1000000,9999999),num_customers,replace=False)

# Assign genders
genders = np.random.choice(['Male','Female'],num_customers)

# Assign gender-matched names
names = [fake.name_male() if g == 'Male' else fake.name_female() for g in genders]

# Generate city and phone details
city_region_map = {
    'Delhi':'North','Jaipur':'North','Chandigarh':'North','Lucknow':'North',
    'Mumbai':'West','Pune':'West','Ahmedabad':'West','Surat':'West',
    'Bangalore':'South','Chennai':'South','Hyderabad':'South','Coimbatore':'South','Madurai':'South','Trichy':'South','Mysuru':'South','Kochi':'South',
    'Kolkata':'East','Patna':'East','Bhopal':'East','Visakhapatnam':'East'
}
indian_cities = list(city_region_map.keys())
cities = np.random.choice(indian_cities,num_customers)
def generate_phone():
  number = '9' + ''.join([str(random.randint(0,9))for _ in range(9)])
  return f"+91 {number}"
phones = [generate_phone() for _ in range(num_customers)]

# Core numeric features
ages = np.random.randint(18,80,num_customers)
balances = np.random.randint(1000,300000,num_customers)
credits = np.random.randint(400,900,num_customers)
tenures = np.random.randint(1,10,num_customers)
salaries = np.random.randint(20000,800000,num_customers)
regions = [city_region_map[c] for c in cities]

# Generate 'IsActive' based on CreditScore and Tenure
def get_is_active(score,tenure):
  if score >= 750 or tenure >= 6:
    return np.random.choice(['Yes','No'],p=[0.9,0.1])
  elif 650 <= score < 750:
    return np.random.choice(['Yes','No'],p=[0.7,0.3])
  elif 550 <= score < 650:
    return np.random.choice(['Yes','No'],p=[0.4,0.6])
  else:
    return np.random.choice(['Yes','No'],p=[0.2,0.8])
is_active = [get_is_active(credits[i],tenures[i])for i in range(num_customers)]

# Build customer dataframe
customers = pd.DataFrame({'CustomerID':customer_ids,
                          'Name':names,
                          'Gender':genders,
                          'Age':ages,
                          'City':cities,
                          'Region':regions,
                          'Tenure':tenures,
                          'Balance':balances,
                          'CreditScore':credits,
                          'IsActive':is_active,
                          'Salary':salaries,
                          'Phone':phones
                          })
# Save dataset to CSV
customers.to_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/' + 'customers.csv',index=False)

In [ ]:
# Generate Transaction Data

num_transactions = 200000
transaction_types = ['Credit','Debit']
channels = ['ATM','Online','Branch','Mobile']

transactions = pd.DataFrame({
    'TransactionID':np.random.choice(range(100000,999999),num_transactions,replace=False),
    'CustomerID':np.random.choice(customers['CustomerID'],num_transactions),
    'Date': [fake.date_between(start_date='-1y',end_date='today')for _ in range(num_transactions)],
    'TransactionType':np.random.choice(transaction_types,num_transactions),
    'Amount':np.random.randint(100,10000,num_transactions),
    'Channel':np.random.choice(channels,num_transactions)
})

# Save Dataset to csv
transactions.to_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/' + 'transactions.csv',index=False)

In [ ]:
# Generate Churn Data

# Convert transaction Date column to datetime
transactions['Date'] = pd.to_datetime(transactions['Date'])

# Calculate last transaction date per customer
last_txn = transactions.groupby('CustomerID')['Date'].max().reset_index()
last_txn.rename(columns={'Date':'LastTransactionDate'},inplace=True)

# Reference Date(today)
today = datetime.now().date()

# Merge with customer dataset
churn_data = customers[['CustomerID']].merge(last_txn,on='CustomerID',how='left')

# Calculate days since last transaction
churn_data['LastTransactionDays']=(today - churn_data['LastTransactionDate'].dt.date).apply(lambda x:x.days)

# Add complaints and number of products for each customer
churn_data['Complaints']=np.random.poisson(1,churn_data.shape[0])
churn_data['Products']=np.random.randint(1,5,churn_data.shape[0])

# More likely to churn if inactive or long since last transaction
def get_churn_probability(days,is_active):
  if is_active == 'No' and days > 90:
    return 0.7
  elif is_active == 'No' and days <= 90:
    return 0.4
  elif is_active == 'Yes' and days > 90:
    return 0.2
  else:
    return 0.1
churn_data['Exited'] = churn_data.apply(lambda x:np.random.choice([1,0]
                                                                  ,p=[get_churn_probability(x['LastTransactionDays'],
                                                                                            customers.loc[customers['CustomerID']==x['CustomerID'],
                                                                                                          'IsActive'].values[0]),
                                                                      1-
                                                                      get_churn_probability(x['LastTransactionDays'],
                                                                                            customers.loc[customers['CustomerID']==x['CustomerID'],
                                                                                                          'IsActive'].values[0])]),
                                        axis=1)

# Save Churn Data
churn_data.to_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/' + 'churn_data.csv',index=False)